In [ ]:
!pip install unsloth "xformers"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of xformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 132.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 60.9 MB/s 

In [ ]:
from unsloth import FastLanguageModel
import transformers

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from huggingface_hub import login
# add your huggingface token here

In [ ]:
#from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "meta-llama/Meta-Llama-3-8B-instruct"
#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
import random

random.seed(42)

# Number of trials
num_trials = 100

# Define the timeline
timeline = []
for i in range(num_trials):
    while True:
        if i < (num_trials / 2):
            bandit_1_reward = random.choices([1, 0], weights=[0.8, 0.2])[0]
            bandit_2_reward = random.choices([1, 0], weights=[0.2, 0.8])[0]
        else:
            bandit_1_reward = random.choices([1, 0], weights=[0.2, 0.8])[0]
            bandit_2_reward = random.choices([1, 0], weights=[0.8, 0.2])[0]

        if not (bandit_1_reward == 0 and bandit_2_reward == 0):
            break

    timeline.append({
        "bandit_1": {"color": "orange", "value": bandit_1_reward},
        "bandit_2": {"color": "blue", "value": bandit_2_reward}
    })

In [ ]:
def generate_seeds(num_seeds=20, seed=42):
    """Generates a list of random seeds.

    Args:
        num_seeds: The number of seeds to generate.
        seed: The initial seed for the random number generator (for reproducibility).

    Returns:
        A list of random integer seeds.
    """
    random.seed(seed)  # Set initial seed for reproducibility
    seeds = [random.randint(1, 100000) for _ in range(num_seeds)]
    return seeds

In [ ]:
seeds=generate_seeds(num_seeds=32)

In [ ]:
import transformers
def create_text_generation_pipeline(model, tokenizer, temperature=1.0, max_new_tokens=1024):
    """
    Creates a text-generation pipeline with the given model and tokenizer.

    Args:
        model: The preloaded model for text generation.
        tokenizer: The corresponding tokenizer.
        temperature (float): Sampling temperature for generation (default: 1.0).
        max_new_tokens (int): Maximum number of tokens to generate (default: 1024).

    Returns:
        A transformers pipeline object for text generation.
    """
    return transformers.pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        trust_remote_code=True,
        pad_token_id=0,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        do_sample=True
    )

# Example usage:
# pipe = create_text_generation_pipeline(model, tokenizer)


In [ ]:
import json
import re

def extract_model_choice(raw_response: str) -> int:
    """
    Extracts choice from model's raw response text
    Handles common variations while maintaining strict validation
    """
    try:
        # First try direct JSON parsing
        response_data = json.loads(raw_response)
        return int(response_data['choice'])

    except json.JSONDecodeError:
        # Fallback: Search for JSON pattern in text
        json_match = re.search(r'{\s*"choice"\s*:\s*[12]\s*}', raw_response)
        if json_match:
            response_data = json.loads(json_match.group())
            return int(response_data['choice'])

    except KeyError:
        pass

    # Final fallback: Find any single digit 1 or 2 in response
    digit_match = re.search(r'\b[12]\b', raw_response)
    if digit_match:
        return int(digit_match.group())

    raise ValueError("No valid choice found in model response")

In [ ]:
def generate(prompt,pipe):
    # Remove any existing print statements from this function
    outputs = pipe(prompt)
    full_text = outputs[0]['generated_text']
    #print(full_text)
    # Extract only the model's choice and reasoning from response
    choice = extract_model_choice(full_text)
    if choice is None:
      return full_text,outputs

    # Return both values
    return choice

In [ ]:
def format_past_trials(past_trials: list) -> str:
    """Formats past trial data for the prompt by listing choice, reward, and cumulative reward."""
    return "".join(
        f"- In trial {trial['trial_num']} you chose {trial['choice']} and you got {trial['reward']}, Cumulative Reward: {trial['cumulative_reward']}\n"
        for trial in past_trials
    )


In [ ]:
def build_slot_prompt(current_trial: int, past_trials: list, total_trials: int) -> str:
    recent_trials = past_trials[-5:] if len(past_trials) > 5 else past_trials
    formatted_trials = format_past_trials(recent_trials)

    return f"""<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>

You are a participant in a two-armed bandit  rewards** over time.
The environment may change unpredictably, and past success does not guarantee future results.

<|eot_id|>

<|start_header_id|>user<|end_header_id|>task. Your goal is to **maximize total

# Task Parameters
- Trial {current_trial} of {total_trials}
- Choose between: [1] or [2]
- Possible outcomes: 1 (success) or 0 (failure)
- Reward probabilities may change at any time

# Recent History
{formatted_trials}

# Required Response
Select your next choice:

<|response_format|>
{{
    "choice": <1|2>
}}

<|critical_instructions|>
- Respond with valid JSON only

<|eot_id|>"""

In [ ]:
import numpy as np
import torch
import random
import transformers

def fix_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    transformers.set_seed(seed)  # For Hugging Face models

In [ ]:
import torch
import gc

# Storage for results
all_results = []

# Run simulation for each seed
for run_id, seed in enumerate(seeds):
    gc.collect()
    torch.cuda.empty_cache()
    fix_seed(seed)  # Ensure reproducibility
    torch.cuda.empty_cache()  # Clear GPU memory before loading model

    # Initialize new model for each seed
    model,tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    model._past = None  # Reset past states if necessary


    history = []
    cumulative_reward = 0
    total_trials = 100
    pipe=create_text_generation_pipeline(model,tokenizer,max_new_tokens=150)
    print(f"\n### Running simulation with seed {seed} (Run {run_id+1}/{len(seeds)}) ###\n")

    # Simulate 100 trials
    for trial in range(1, total_trials + 1):
        current_trial_data = timeline[trial - 1]  # Ensure `timeline` is defined
        prompt_model = build_slot_prompt(trial, history, total_trials)
        bandit_1_value = current_trial_data["bandit_1"]["value"]
        bandit_2_value = current_trial_data["bandit_2"]["value"]
        #print(f"this is {prompt_model}")
        max_retries = 5
        for attempt in range(max_retries):
            model_choice = generate(prompt_model,pipe)
            if model_choice in {1, 2}:
                break
        else:
            # Fallback to random choice after failures
            model_choice=None

        # Determine reward
        reward = bandit_1_value if model_choice == 1 else (bandit_2_value if model_choice == 2 else 0)
        cumulative_reward += reward
        #outputs=generate_test(prompt_model,pipe)
        #print(f"this is whole output {outputs}")

        history.append({
            "trial_num": trial,
            "choice": model_choice,
            "reward": reward,
            "cumulative_reward": cumulative_reward,
            #"reasoning": model_reasoning,
        })

        print(f"Trial {trial}: "
              f"Choice {model_choice}, "
              f"Reward {reward}, "
              #f"Reasoning {model_reasoning} "
              f"Total {cumulative_reward}")

    all_results.append({
        "seed": seed,
        "run_id": run_id,
        "history": history,
        "final_score": cumulative_reward
    })

    print(f"\nFinal score for seed {seed}: {cumulative_reward}\n")

    # Cleanup: delete model and clear memory
    del model
    gc.collect()
    torch.cuda.empty_cache()


==((====))==  Unsloth 2025.6.8: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

In [ ]:
import pandas as pd

# Create an empty DataFrame
history_pd = pd.DataFrame(columns=['trial', 'choice', 'reward', 'score'])

# Initialize score
cumulative_score = 0

# Iterate through the history list
for i, trial_data in enumerate(history):
    # Calculate cumulative score
    cumulative_score += trial_data['reward']

    # Append data to the DataFrame
    history_pd.loc[len(history_pd)] = [i + 1, trial_data['choice'], trial_data['reward'],cumulative_score]

In [ ]:
all_results[0]['history']

In [ ]:
# Assuming all_results is a list of dictionaries
all_dfs = []

for i in range(len(all_results)):  # Iterate through each model run
    model_id = all_results[i]['seed']  # Retrieve model_id (seed)
    history = all_results[i]['history']  # Retrieve history

    df = pd.DataFrame(history)  # Convert history to DataFrame
    df.insert(0, 'trial', range(1, len(df) + 1))  # Add trial number
    df['model_id'] = model_id  # Add model_id

    df.rename(columns={'cumulative_reward': 'score'}, inplace=True)  # Rename column

    all_dfs.append(df)  # Append to list

# Concatenate all DataFrames into one
final_df = pd.concat(all_dfs, ignore_index=True)

print(final_df)


In [ ]:
final_df

In [ ]:
final_df.to_csv('llama-3.1_32.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot value over trial number
plt.figure(figsize=(10, 5))
sns.lineplot(data=history_pd, x="trial", y="score", marker="o")

# Labels and title
plt.xlabel("Trial Number")
plt.ylabel("Score")
plt.title("Score Over Trials")
plt.grid(True)

# Show plot
plt.show()


In [ ]:
# @title trial vs reward

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.scatter(history_pd['trial'],history_pd["reward"], alpha=0.5, label="Individual Choices")

plt.xlabel("Trial Number")
plt.ylabel("Value (0 or 1)")
plt.title("Scatter Plot of Choices Over Time")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# @title choice

from matplotlib import pyplot as plt
import seaborn as sns
history_pd.groupby('choice').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title averaged choosed
# Convert 'response' column to count bandit choices (1 if bandit_1, 0 if bandit_2)
history_pd["bandit_2_chosen"] = (history_pd["choice"] == "2").astype(int)
history_pd["bandit_1_chosen"] = (history_pd["choice"] == "1").astype(int)


# Compute rolling averages (smoothing with a 10-trial window)
history_pd["bandit_1_avg"] = history_pd["bandit_1_chosen"].rolling(window=10, min_periods=1).mean()
history_pd["bandit_2_avg"] = history_pd["bandit_2_chosen"].rolling(window=10, min_periods=1).mean()

# Plot
plt.figure(figsize=(10, 5))
plt.plot(history_pd.index, history_pd["bandit_1_avg"], label="Bandit 1", color="blue")
plt.plot(history_pd.index, history_pd["bandit_2_avg"], label="Bandit 2", color="red")
# Add a dotted line on the x-axis to indicate switch
plt.axvline(x=50, color='black', linestyle='--', linewidth=1.5, label="Switch Point (Trial 50)")

plt.xlabel("Trial Number")
plt.ylabel("Proportion Chosen")
plt.title("Trend of Choosing Bandit 1 vs Bandit 2 Over Time")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
prompt = "You will be presented with triplets of objects, which will be assigned to the keys H, Y, and E.\n" \
  "In each trial, please indicate which object you think is the odd one out by pressing the corresponding key.\n" \
  "In other words, please choose the object that is the least similar to the other two.\n\n" \
  "H: plant, Y: chainsaw, and E: periscope. You press <<H>>.\n" \
  "H: tostada, Y: leaf, and E: sail. You press <<H>>.\n" \
  "H: clock, Y: crystal, and E: grate. You press <<Y>>.\n" \
  "H: barbed wire, Y: kale, and E: sweater. You press <<E>>.\n" \
  "H: raccoon, Y: toothbrush, and E: ice. You press <<"

print(prompt)

In [ ]:
print(prompt)

In [ ]:
for i, trial_data in enumerate(timeline, start=1):
        print (
            f"Trial {i}: Bandit 1 ({trial_data['bandit_1']['color']}) → {trial_data['bandit_1']['value']}, "
            f"Bandit 2 ({trial_data['bandit_2']['color']}) → {trial_data['bandit_2']['value']}\n"
        )


In [ ]:
### LLM ###
model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = model_name,
  max_seq_length = 2024,
  dtype = None,
  load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

In [ ]:
from transformers import pipeline
text_generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
prompt = "Think carefully before answer.I have tomatoes, basil, and cheese at home. What can I cook for dinner?"
sequences = text_generator(prompt, num_return_sequences=1)
for seq in sequences:
    print(seq['generated_text'])

In [ ]:
# Configure trial data
current_trial = 5
past_results = [
    "Trial 1: Chose 1 → 1 point",
    "Trial 2: Chose 2 → 0 points",
    "Trial 3: Chose 2 → 0 points",  # Note: Typo in original preserved
    "Trial 4: Chose 1 → 1 point"
]

# Generate prompt
prompt_test = build_slot_prompt(current_trial, past_results)
print(prompt_test)  # Send this to your model